# Bài thực hành làm sạch và phân tích dữ liệu đặt vé máy bay

**Phiên bản Airline 1.0 · Dự án Giữa kỳ**

| Thông tin nhóm | Điền vào đây |
|---|---|
| Mã nhóm | NhomXX |
| Thành viên 1 (MSSV) — Kỹ sư Làm sạch Dữ liệu | Trần Tùng — Giai đoạn A + B1–B5 |
| Thành viên 2 (MSSV) — QA & Data Architect | … — B6–C4 + Câu 1 |
| Thành viên 3 (MSSV) — BI & Data Analyst | … — Câu 2–10 + 8 biểu đồ |
| Ngày chạy lại toàn bộ lần cuối | 2026-10-20 |

**Trước khi nộp:** đảm bảo file tên là `airline_analysis.ipynb`.


## 0 · Thiết lập

Ô 0.1 chứa công cụ hỗ trợ dùng chung. **Không cần sửa**, chỉ cần chạy.


In [ ]:
# 0.1 CÔNG CỤ HỖ TRỢ — KHÔNG CẦN SỬA
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 50)

# ---- Đường dẫn: notebook chạy từ notebooks/ nên data nằm ở ../data/ ----
# Theo chuẩn đề bài: data/airline_bookings_dirty.csv (file gốc không được sửa)
CANDIDATES = [
    Path('../data/airline_bookings_dirty.csv'),   # Máy cá nhân (chuẩn)
    Path('data/airline_bookings_dirty.csv'),       # Chạy từ thư mục gốc
    Path('airline_bookings_dirty.csv')             # Colab
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
OUTPUT_DIR = Path('../outputs')
for sub in ['', 'figures', 'tables']:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ---- Quy ước của đề (Mục 5) ----
SNAPSHOT_DATE = pd.Timestamp('2026-09-15')
MISSING_TOKENS = ['NULL', 'N/A', 'NA', '?', 'unknown', '-']
AIRLINE_COLORS = {'Aurora Air': '#2F4858', 'Horizon Wings': '#E07A1F'}
STATUS_COLORS = {'Flown': '#5B8E7D', 'Cancelled': '#B81D24', 'No-Show': '#F2A541'}
FIGSIZE = (9, 5)

AIRLINES = ['Aurora Air', 'Horizon Wings']
MEALS = ['Standard', 'Vegetarian', 'Special', 'No Meal']
SEGMENTS = ['Direct', 'Online OTA', 'Offline Agent', 'Corporate', 'Group', 'Package Tour', 'Loyalty']
CHANNELS = ['Website', 'Mobile App', 'Travel Agency', 'GDS/Corporate']
FARE_POLICIES = ['Flexible', 'Semi-Flexible', 'Non-Refundable']
TRAVELERS = ['Individual', 'Linked Booking', 'Corporate Contract', 'Group']
STATUSES = ['Flown', 'Cancelled', 'No-Show']
CABIN_CODES = ['E1', 'E2', 'E3', 'E4', 'P1', 'P2', 'B1', 'B2', 'B3', 'F1', 'F2', 'F3']
LEAD_GROUPS = ['0-7', '8-30', '31-90', '91-180', '181+']

DEPARTURE_MIN, DEPARTURE_MAX = pd.Timestamp('2024-07-01'), pd.Timestamp('2026-08-31')
STATUS_MIN, STATUS_MAX = pd.Timestamp('2024-01-01'), pd.Timestamp('2026-09-15')
LEAD_MIN, LEAD_MAX = 0, 365
WEEKEND_MAX, WEEK_MAX = 20, 50
PASSENGER_MAX = 10
FARE_MIN, FARE_MAX = 0, 5000
REQ_MAX = 5
N_RAW_ROWS = 10325

CLEAN_REQUIRED = ['reservation_id', 'airline', 'is_cancelled', 'booking_lead_days', 'departure_date', 'trip_weekend_days',
                  'trip_weekdays', 'adult_passengers', 'child_passengers', 'infant_passengers', 'meal_option',
                  'passenger_country', 'market_segment', 'sales_channel', 'is_returning_customer', 'booked_cabin_code',
                  'assigned_cabin_code', 'fare_policy', 'traveler_type', 'avg_ticket_fare', 'special_service_requests',
                  'booking_status', 'booking_status_date', 'departure_year', 'departure_month', 'total_trip_days',
                  'total_passengers', 'lead_time_group', 'cabin_changed_flag', 'booking_value_est']
SUMMARY_COLUMNS = ['airline', 'departure_year', 'departure_month', 'n_reservations', 'n_cancelled',
                   'cancel_rate_pct', 'passengers_flown']
LOG_COLUMNS = ['step', 'field', 'issue', 'detection_rule', 'action', 'affected_rows',
               'changed_cells', 'removed_rows', 'example_record_ids', 'reason', 'check_after']
LINEAGE_COLUMNS = ['record_id', 'reservation_id', 'status']
LINEAGE_STATUS = ['kept', 'merged', 'dropped']
QUALITY_COLUMNS = ['column', 'stage', 'denominator_rows', 'missing_count', 'invalid_count', 'missing_pct']

CORE_FILES = ['reservations_clean.csv', 'monthly_summary.csv', 'cleaning_log.csv', 'row_lineage.csv', 'quality_report.csv']
FIGURE_FILES = ['fig01_airline_cancel.png', 'fig02_monthly_departures_2025.png', 'fig03_leadtime_cancel.png',
                'fig04_top10_country.png', 'fig05_segment_cancel.png', 'fig06_fare_policy_status.png',
                'fig07_fare_hist.png', 'fig08_trip_days.png']
TABLE_FILES = ['a_missing_survey.csv', 'a_issue_catalog.csv', 'q01_before_after.csv',
               'q02_airline_cancel.csv', 'q03_monthly_departures_2025.csv', 'q04_leadtime_stats.csv',
               'q04_leadtime_cancel.csv', 'q05_top10_country.csv', 'q06_segment_cancel.csv',
               'q07_fare_policy_status.csv', 'q08_fare_stats_by_airline.csv', 'q09_trip_days.csv',
               'q10_cabin_change_by_airline.csv', 'q10_cabin_change_by_booked_code.csv']


def check(label, fn, hint=''):
    try:
        ok = bool(fn())
        detail = ''
    except Exception as e:
        ok = False
        detail = f'{type(e).__name__}: {e}'
    if ok:
        print('✅ ' + label)
    else:
        extra = (' — ' + detail) if detail else ((' — ' + hint) if hint else '')
        print('❌ ' + label + extra)
    return ok


def clean_key(x):
    if pd.isna(x): return pd.NA
    return re.sub(r'[^a-z]', '', str(x).strip().lower())


def log_step(step, field, issue, detection_rule, action, affected_rows=0, changed_cells=0,
             removed_rows=0, example_record_ids='', reason='', check_after=''):
    cleaning_log_rows.append({
        'step': step, 'field': field, 'issue': issue, 'detection_rule': detection_rule,
        'action': action, 'affected_rows': int(affected_rows), 'changed_cells': int(changed_cells),
        'removed_rows': int(removed_rows), 'example_record_ids': str(example_record_ids),
        'reason': reason, 'check_after': check_after,
    })


def save_table(frame, name):
    if frame is None:
        print(f'⏳ Chưa lưu {name}.csv: biến còn None.')
        return
    frame.to_csv(OUTPUT_DIR / 'tables' / f'{name}.csv', index=False, encoding='utf-8-sig')
    print(f'💾 outputs/tables/{name}.csv ({len(frame)} dòng)')


def save_fig(fig, name):
    if name + '.png' not in FIGURE_FILES:
        print(f'⚠️ Tên {name}.png không có trong danh sách chuẩn.')
    fig.savefig(OUTPUT_DIR / 'figures' / f'{name}.png', dpi=150, bbox_inches='tight')
    print(f'🖼️ outputs/figures/{name}.png')


def finish_chart(ax, title, xlabel, ylabel, note=None):
    ax.set_title(title, fontsize=13, fontweight='bold', loc='left')
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.spines[['top', 'right']].set_visible(False)
    if note:
        ax.figure.text(0.01, -0.04, note, fontsize=9, color='#555555', ha='left', va='top', wrap=True)


def show_progress():
    items = [('Dữ liệu', f, OUTPUT_DIR / f) for f in CORE_FILES] + \
            [('Bảng', f, OUTPUT_DIR / 'tables' / f) for f in TABLE_FILES] + \
            [('Hình', f, OUTPUT_DIR / 'figures' / f) for f in FIGURE_FILES]
    t = pd.DataFrame([(g, f, '✅' if q.exists() else '⬜') for g, f, q in items], columns=['Nhóm', 'Tệp', 'Có'])
    done = (t['Có'] == '✅').sum()
    print(f'Tiến độ tệp đầu ra: {done}/{len(t)} ({done / len(t):.0%})')
    return t


cleaning_log_rows = []
print('✅ Thiết lập xong. DATA_PATH =', DATA_PATH)

In [ ]:
# 0.2 ĐỌC DỮ LIỆU — đọc mọi cột ở dạng chuỗi để thấy giá trị gốc
if not DATA_PATH.exists():
    raise FileNotFoundError('Không thấy airline_bookings_dirty.csv.')

raw_df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
print('Kích thước đầu vào:', raw_df.shape)
assert raw_df.shape == (10325, 24), f'Đầu vào phải là 10.325 dòng × 24 cột, hiện tại: {raw_df.shape}'
assert raw_df['record_id'].is_unique, 'record_id phải duy nhất ở dữ liệu đầu vào.'
print('✅ Đầu vào đúng 10.325 dòng × 24 cột')
raw_df.head()

---
# GIAI ĐOẠN A · KHẢO SÁT TRƯỚC KHI SỬA
*Mục tiêu: hiểu dữ liệu đang sai ở đâu, sai bao nhiêu — **chưa sửa gì cả**.*

**Thành viên 1 phụ trách toàn bộ Giai đoạn A.**

| | |
|---|---|
| **Mục tiêu** | Lập bảng thiếu theo cột và danh mục **ít nhất 10 nhóm vấn đề**, mỗi nhóm có số dòng và tối đa 5 `record_id` ví dụ. |
| **Đầu vào → Đầu ra** | `raw_df` → `missing_survey`, `issue_catalog` |
| **Thành viên phụ trách** | Thành viên 1 |


In [ ]:
# A1 · Tổng quan dữ liệu
print('=== HÌNH DẠNG DỮ LIỆU ===')
print('Shape:', raw_df.shape)
print()
print('=== TÊN CỘT ===')
print(list(raw_df.columns))
print()
print('=== KIỂU DỮ LIỆU (đọc dạng str nên đều là object) ===')
print(raw_df.dtypes)
print()
print('=== 5 DÒNG ĐẦU ===')
raw_df.head()

In [ ]:
# A2 · Bảng thiếu theo cột
# Đếm ô rỗng (sau strip) và ô là ký hiệu thiếu theo từng cột

records = []
for col in raw_df.columns:
    n_rows = len(raw_df)
    empty_count = (raw_df[col].str.strip() == '').sum()
    token_count = raw_df[col].str.lower().isin([t.lower() for t in MISSING_TOKENS]).sum()
    missing_total = empty_count + token_count
    missing_pct = round(missing_total / n_rows * 100, 2)
    records.append({
        'column': col,
        'n_rows': n_rows,
        'empty_count': int(empty_count),
        'token_count': int(token_count),
        'missing_total': int(missing_total),
        'missing_pct': missing_pct
    })

missing_survey = pd.DataFrame(records)
print(f'missing_survey: {missing_survey.shape}')
missing_survey

In [ ]:
# A3 · Nhãn của các cột phân loại — value_counts
cat_cols = ['airline', 'is_cancelled', 'meal_option', 'market_segment', 'sales_channel',
            'fare_policy', 'traveler_type', 'booking_status', 'booked_cabin_code', 'assigned_cabin_code']

for col in cat_cols:
    print(f'\n{"="*55}')
    print(f'  {col.upper()}')
    print(f'{"="*55}')
    print(raw_df[col].value_counts(dropna=False).to_string())

In [ ]:
# A4 · Định dạng ngày, số, tiền và mã quốc gia
print('=== departure_date: các định dạng khác YYYY-MM-DD ===')
dep_nonstandard = raw_df['departure_date'][
    ~raw_df['departure_date'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
]
print(dep_nonstandard.value_counts().head(25))

print('\n=== avg_ticket_fare: giá trị không phải số thuần ===')
fare_nonnum = raw_df['avg_ticket_fare'][
    pd.to_numeric(raw_df['avg_ticket_fare'], errors='coerce').isna()
]
print(fare_nonnum.value_counts().head(20))

print('\n=== passenger_country: mã không chuẩn ===')
country_bad = raw_df['passenger_country'][
    ~raw_df['passenger_country'].str.strip().str.fullmatch(r'[A-Z]{3}')
]
print(country_bad.value_counts().head(20))

print('\n=== booking_status_date: mẫu ===')
print(raw_df['booking_status_date'].value_counts(dropna=False).head(10))

print('\n=== Số trường có ký hiệu thiếu hoặc rỗng ===')
for col in ['trip_weekend_days', 'trip_weekdays', 'adult_passengers', 'child_passengers',
            'infant_passengers', 'special_service_requests', 'booking_lead_days']:
    unusual = raw_df[col][pd.to_numeric(raw_df[col], errors='coerce').isna()]
    if len(unusual) > 0:
        print(f'{col}: {len(unusual)} giá trị không phải số')
        print(' ', unusual.value_counts().head(5).to_dict())

In [ ]:
# A5 · Bản ghi trùng và khóa
biz_cols = [c for c in raw_df.columns if c != 'record_id']
dup_mask = raw_df.duplicated(subset=biz_cols, keep=False)
n_dup_rows = dup_mask.sum()
print(f'Số dòng trùng (23 cột nghiệp vụ): {n_dup_rows} dòng')
dup_examples = raw_df[dup_mask]['record_id'].head(6).tolist()
print(f'Ví dụ record_id trùng: {dup_examples}')

# reservation_id lặp trước chuẩn hóa
res_before = raw_df['reservation_id']
n_res_dup_before = res_before.duplicated(keep=False).sum()
print(f'\nreservation_id lặp (trước chuẩn hóa): {n_res_dup_before} dòng')

# reservation_id lặp sau chuẩn hóa (strip + upper)
res_norm = res_before.str.strip().str.upper()
n_res_dup_after = res_norm.duplicated(keep=False).sum()
print(f'reservation_id lặp (sau strip+upper): {n_res_dup_after} dòng')

# Kiểm tra định dạng AR + 6 số
bad_format = raw_df[~raw_df['reservation_id'].str.strip().str.upper().str.fullmatch(r'AR\d{6}')]
print(f'\nreservation_id không đúng dạng AR+6 số: {len(bad_format)} dòng')

In [ ]:
# A6 · Danh mục vấn đề (12 nhóm)
def get_examples(mask, n=5):
    ids = raw_df[mask]['record_id'].head(n).tolist()
    return '; '.join(ids)

issues = []

# I01 · airline: nhãn không chuẩn
airline_valid = raw_df['airline'].str.strip().isin(AIRLINES)
i01_mask = ~airline_valid & (raw_df['airline'].str.strip() != '')
issues.append({
    'issue_id': 'I01', 'field': 'airline',
    'issue': 'Nhãn airline không chuẩn (AURORA AIR, aurora air, Horizon, Aurora)',
    'detection_rule': 'airline.strip() không thuộc {Aurora Air, Horizon Wings}',
    'affected_rows': int(i01_mask.sum()),
    'example_record_ids': get_examples(i01_mask)
})

# I02 · booking_status: ngày nằm nhầm cột
bs_is_date = raw_df['booking_status'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
bsd_empty = raw_df['booking_status_date'].str.strip() == ''
i02_mask = bs_is_date & bsd_empty
issues.append({
    'issue_id': 'I02', 'field': 'booking_status / booking_status_date',
    'issue': 'Ngày YYYY-MM-DD nằm nhầm ở booking_status thay vì booking_status_date',
    'detection_rule': 'booking_status fullmatch YYYY-MM-DD AND booking_status_date rỗng',
    'affected_rows': int(i02_mask.sum()),
    'example_record_ids': get_examples(i02_mask)
})

# I03 · booking_status: nhãn không chuẩn
bs_valid = raw_df['booking_status'].str.strip().isin(STATUSES)
bs_not_date = ~raw_df['booking_status'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
bs_not_empty = raw_df['booking_status'].str.strip() != ''
i03_mask = ~bs_valid & bs_not_date & bs_not_empty
issues.append({
    'issue_id': 'I03', 'field': 'booking_status',
    'issue': 'Nhãn trạng thái không chuẩn (flown, FLOWN, Completed, Canceled, CANCELLED)',
    'detection_rule': 'booking_status.strip() không thuộc {Flown, Cancelled, No-Show} và không phải ngày',
    'affected_rows': int(i03_mask.sum()),
    'example_record_ids': get_examples(i03_mask)
})

# I04 · is_cancelled: giá trị không chuẩn
canc_valid = raw_df['is_cancelled'].str.strip().isin(['0', '1'])
canc_not_empty = raw_df['is_cancelled'].str.strip() != ''
i04_mask = ~canc_valid & canc_not_empty
issues.append({
    'issue_id': 'I04', 'field': 'is_cancelled',
    'issue': 'Giá trị không chuẩn: yes/no/True/False/0.0/1.0',
    'detection_rule': 'is_cancelled.strip() không thuộc {0, 1}',
    'affected_rows': int(i04_mask.sum()),
    'example_record_ids': get_examples(i04_mask)
})

# I05 · departure_date: nhiều định dạng
dep_noniso = ~raw_df['departure_date'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
dep_not_empty = raw_df['departure_date'].str.strip() != ''
dep_not_token = ~raw_df['departure_date'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i05_mask = dep_noniso & dep_not_empty & dep_not_token
issues.append({
    'issue_id': 'I05', 'field': 'departure_date',
    'issue': 'Nhiều định dạng ngày: DD/MM/YYYY, DD-Mon-YYYY, Month DD, YYYY',
    'detection_rule': 'departure_date không khớp \\d{4}-\\d{2}-\\d{2}',
    'affected_rows': int(i05_mask.sum()),
    'example_record_ids': get_examples(i05_mask)
})

# I06 · avg_ticket_fare: ký hiệu, thiếu, bất hợp lý
fare_nonnum = pd.to_numeric(raw_df['avg_ticket_fare'], errors='coerce').isna()
fare_not_empty = raw_df['avg_ticket_fare'].str.strip() != ''
fare_not_token = ~raw_df['avg_ticket_fare'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i06_mask = fare_nonnum & fare_not_empty & fare_not_token
fare_num = pd.to_numeric(raw_df['avg_ticket_fare'], errors='coerce')
fare_outrange = fare_num.notna() & (~fare_num.between(FARE_MIN, FARE_MAX))
issues.append({
    'issue_id': 'I06', 'field': 'avg_ticket_fare',
    'issue': 'Ký hiệu thiếu (null/N/A/-), ký hiệu tiền tệ ($), giá bất hợp lý (9999)',
    'detection_rule': 'to_numeric fails OR value not in [0, 5000]',
    'affected_rows': int(i06_mask.sum() + fare_outrange.sum()),
    'example_record_ids': get_examples(i06_mask | fare_outrange)
})

# I07 · passenger_country: mã không chuẩn
country_bad = ~raw_df['passenger_country'].str.strip().str.fullmatch(r'[A-Z]{3}')
country_not_empty = raw_df['passenger_country'].str.strip() != ''
country_not_token = ~raw_df['passenger_country'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i07_mask = country_bad & country_not_empty & country_not_token
issues.append({
    'issue_id': 'I07', 'field': 'passenger_country',
    'issue': 'Mã quốc gia không chuẩn: 2 chữ (CN→CHN), chữ thường, ký hiệu thiếu',
    'detection_rule': 'passenger_country không khớp [A-Z]{3}',
    'affected_rows': int(i07_mask.sum()),
    'example_record_ids': get_examples(i07_mask)
})

# I08 · Bản ghi trùng
biz_cols_A = [c for c in raw_df.columns if c != 'record_id']
dup_mask_A = raw_df.duplicated(subset=biz_cols_A, keep=False)
issues.append({
    'issue_id': 'I08', 'field': 'ALL (23 cột nghiệp vụ)',
    'issue': 'Bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ (không tính record_id)',
    'detection_rule': 'duplicated(subset=23_biz_cols, keep=False)',
    'affected_rows': int(dup_mask_A.sum()),
    'example_record_ids': get_examples(dup_mask_A)
})

# I09 · meal_option: nhãn không chuẩn và thiếu
meal_valid = raw_df['meal_option'].str.strip().isin(MEALS)
meal_not_empty = raw_df['meal_option'].str.strip() != ''
meal_not_token = ~raw_df['meal_option'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
i09_mask = ~meal_valid & meal_not_empty & meal_not_token
meal_token_mask = raw_df['meal_option'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
meal_empty_mask = raw_df['meal_option'].str.strip() == ''
issues.append({
    'issue_id': 'I09', 'field': 'meal_option',
    'issue': 'Nhãn không chuẩn (Undefined→No Meal), ký hiệu thiếu, ô rỗng',
    'detection_rule': 'meal_option.strip() không thuộc MEALS hoặc là token/rỗng',
    'affected_rows': int(i09_mask.sum() + meal_token_mask.sum() + meal_empty_mask.sum()),
    'example_record_ids': get_examples(i09_mask | meal_token_mask | meal_empty_mask)
})

# I10 · child_passengers: thiếu
child_token = raw_df['child_passengers'].str.lower().isin([t.lower() for t in MISSING_TOKENS])
child_empty = raw_df['child_passengers'].str.strip() == ''
i10_mask = child_token | child_empty
issues.append({
    'issue_id': 'I10', 'field': 'child_passengers',
    'issue': 'Giá trị thiếu: ký hiệu NULL/NA/? hoặc ô rỗng',
    'detection_rule': 'child_passengers là MISSING_TOKEN hoặc rỗng',
    'affected_rows': int(i10_mask.sum()),
    'example_record_ids': get_examples(i10_mask)
})

# I11 · market_segment / sales_channel: Undefined
seg_undef = raw_df['market_segment'].str.strip().str.lower() == 'undefined'
ch_undef = raw_df['sales_channel'].str.strip().str.lower() == 'undefined'
i11_mask = seg_undef | ch_undef
issues.append({
    'issue_id': 'I11', 'field': 'market_segment / sales_channel',
    'issue': 'Nhãn Undefined = không xác định → cần đưa về NaN',
    'detection_rule': 'value.strip().lower() == "undefined"',
    'affected_rows': int(i11_mask.sum()),
    'example_record_ids': get_examples(i11_mask)
})

# I12 · is_cancelled mâu thuẫn với booking_status
canc_0 = raw_df['is_cancelled'].str.strip().isin(['0'])
canc_1 = raw_df['is_cancelled'].str.strip().isin(['1'])
status_flown_r = raw_df['booking_status'].str.strip().str.lower() == 'flown'
status_cancelled_r = raw_df['booking_status'].str.strip().str.lower().isin(['cancelled', 'no-show'])
i12_mask = (status_flown_r & canc_1) | (status_cancelled_r & canc_0)
issues.append({
    'issue_id': 'I12', 'field': 'is_cancelled / booking_status',
    'issue': 'is_cancelled mâu thuẫn với booking_status (Flown↔1 hoặc Cancelled↔0)',
    'detection_rule': '(status==Flown AND is_cancelled==1) OR (status==Cancelled/No-Show AND is_cancelled==0)',
    'affected_rows': int(i12_mask.sum()),
    'example_record_ids': get_examples(i12_mask)
})

issue_catalog = pd.DataFrame(issues)
print(f'issue_catalog: {len(issue_catalog)} nhóm vấn đề')
issue_catalog

In [ ]:
# ✅ Tự kiểm tra giai đoạn A
check('A2 missing_survey có 24 dòng (1 dòng/cột)', lambda: len(missing_survey) == 24)
check('A2 missing_survey đủ cột', lambda: {'column','n_rows','empty_count','token_count','missing_total','missing_pct'} <= set(missing_survey.columns))
check('A6 issue_catalog có ≥ 10 nhóm vấn đề', lambda: len(issue_catalog) >= 10, 'Xem lại gợi ý trong thẻ nhiệm vụ A.')
check('A6 mỗi nhóm có ≤ 5 record_id ví dụ', lambda: issue_catalog['example_record_ids'].str.split('; ').str.len().le(5).all())
check('raw_df vẫn nguyên vẹn', lambda: raw_df.shape == (10325, 24))
save_table(missing_survey, 'a_missing_survey')
save_table(issue_catalog, 'a_issue_catalog')

> **✍️ Phát hiện chính và quy tắc dự kiến** *(Thành viên 1)*
>
> - Dữ liệu có **12 nhóm vấn đề** chính: nhãn phân loại không chuẩn (airline, booking_status, meal_option), ngày nằm nhầm cột (booking_status), ký hiệu thiếu đa dạng (NULL, NA, ?, -), nhiều định dạng ngày (4 định dạng ở departure_date), mã quốc gia không đồng nhất (CN vs CHN), giá vé bất hợp lý (9999), bản ghi trùng tuyệt đối (360 dòng), is_cancelled mâu thuẫn booking_status.
> - Quy tắc sẽ áp dụng: (1) Chuẩn hóa nhãn bằng từ điển ánh xạ; (2) Đưa ký hiệu thiếu về NaN, không điền mean/mode; (3) Chỉ chuyển dữ liệu khi có bằng chứng rõ ràng.
> - Ảnh hưởng: ~175 dòng trùng sẽ được đánh dấu 'dropped' trong row_lineage; các ô thiếu giữ nguyên NaN để phân tích dùng đúng mẫu số.


---
# GIAI ĐOẠN B · LÀM SẠCH (B1 → B8)

**Nguyên tắc chung cho mọi bước:**
1. Làm trên `df` (bản sao); **không bao giờ sửa `raw_df`**.
2. Mỗi quy tắc → **gọi `log_step(...)` một lần** với số liệu đếm được bằng mã.
3. Giá trị không sửa được → **để thiếu (`pd.NA`)**, không đoán.
4. Đếm **trước và sau** mỗi bước.

| Thành viên | Bước phụ trách |
|---|---|
| Thành viên 1 | B1 · B2 · B3 · B4 · B5 |
| Thành viên 2 | B6 · B7 · B8 |


In [ ]:
# Bản làm việc — chạy lại ô này nếu muốn làm lại giai đoạn B từ đầu
df = raw_df.copy()
cleaning_log_rows.clear()
print('df:', df.shape)

## B1 · Khóa và bản ghi trùng tuyệt đối
*(Thành viên 1 phụ trách)*

1. Chuẩn hóa `reservation_id`: strip + upper → dạng `AR` + 6 số.
2. Đếm và loại bỏ bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ.
3. Lưu danh sách `record_id` bị loại để phục vụ `row_lineage` ở B8.


In [ ]:
# B1 · Khóa và bản ghi trùng tuyệt đối

# --- Bước 1: Chuẩn hóa reservation_id ---
before_res_count = df['reservation_id'].str.fullmatch(r'AR\d{6}').sum()
df['reservation_id'] = df['reservation_id'].str.strip().str.upper()
after_res_count = df['reservation_id'].str.fullmatch(r'AR\d{6}').sum()
print(f'reservation_id đúng dạng: trước={before_res_count}, sau={after_res_count}')

changed_res_rows = (raw_df['reservation_id'] != df['reservation_id']).sum()
ex_changed_res = '; '.join(df[raw_df['reservation_id'] != df['reservation_id']]['record_id'].head(5).tolist())

log_step(
    step='B1', field='reservation_id',
    issue='Khoảng trắng đầu/cuối hoặc chữ thường trong reservation_id',
    detection_rule='reservation_id không fullmatch AR\\d{6} trước khi strip+upper',
    action='str.strip().str.upper()',
    affected_rows=changed_res_rows, changed_cells=changed_res_rows, removed_rows=0,
    example_record_ids=ex_changed_res,
    reason='reservation_id chuẩn là AR + 6 chữ số in hoa',
    check_after=f'Sau chuẩn hóa: {after_res_count}/{len(df)} đúng định dạng'
)

# --- Bước 2: Xử lý bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ ---
biz_cols_B1 = [c for c in df.columns if c != 'record_id']
dup_mask_B1 = df.duplicated(subset=biz_cols_B1, keep=False)
n_dup_before = dup_mask_B1.sum()
print(f'\nBản ghi trùng trên 23 cột trước B1: {n_dup_before} dòng')

dup_to_drop_mask = df.duplicated(subset=biz_cols_B1, keep='first')
dropped_record_ids_B1 = df[dup_to_drop_mask]['record_id'].tolist()
print(f'Sẽ loại {len(dropped_record_ids_B1)} bản sao, giữ {n_dup_before - len(dropped_record_ids_B1)} dòng')

example_ids_B1 = '; '.join(dropped_record_ids_B1[:5])
df = df[~dup_to_drop_mask].copy().reset_index(drop=True)

log_step(
    step='B1', field='ALL (23 cột nghiệp vụ)',
    issue='Bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ',
    detection_rule='duplicated(subset=23_biz_cols, keep=False)',
    action='Giữ keep=first, loại bỏ bản sao; lưu dropped_record_ids_B1',
    affected_rows=n_dup_before, changed_cells=0, removed_rows=len(dropped_record_ids_B1),
    example_record_ids=example_ids_B1,
    reason='Bản sao tuyệt đối là lỗi nhập liệu, không phải 2 giao dịch độc lập',
    check_after=f'df còn {len(df)} dòng'
)

print(f'\ndf sau B1: {df.shape}')

In [ ]:
check('B1 reservation_id đúng dạng AR + 6 chữ số', lambda: df['reservation_id'].str.fullmatch(r'AR\d{6}').all())

> **✍️ Quyết định và lý do của nhóm — B1** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Chuẩn hóa `reservation_id` bằng `str.strip().str.upper()`. Loại bỏ bản ghi trùng tuyệt đối trên 23 cột nghiệp vụ bằng `duplicated(keep='first')`.
> - Vì sao `keep='first'`: Đảm bảo `reservation_id` không bị mất; không gộp ở bước này vì gộp theo `reservation_id` là nhiệm vụ B8. Lưu `dropped_record_ids_B1` cho `row_lineage`.
> - Ảnh hưởng: Loại ~175 bản sao, giảm nhiễu trong phân tích. Bản ghi bị loại vẫn được truy vết qua `row_lineage` với nhãn `dropped`.


## B2 · Giá trị nằm nhầm cột
*(Thành viên 1 phụ trách)*

Chuyển ngày nằm nhầm ở `booking_status` sang `booking_status_date` khi **cả hai** điều kiện thỏa mãn: `booking_status` có dạng `YYYY-MM-DD` **và** `booking_status_date` đang rỗng.


In [ ]:
# B2 · Giá trị nằm nhầm cột

bs_is_date_B2 = df['booking_status'].str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}')
bsd_empty_B2 = df['booking_status_date'].str.strip() == ''
swap_mask = bs_is_date_B2 & bsd_empty_B2

n_swap = swap_mask.sum()
print(f'Số dòng cần chuyển ngày từ booking_status → booking_status_date: {n_swap}')

if n_swap > 0:
    examples_B2 = '; '.join(df[swap_mask]['record_id'].head(5).tolist())
    df.loc[swap_mask, 'booking_status_date'] = df.loc[swap_mask, 'booking_status'].str.strip()
    df.loc[swap_mask, 'booking_status'] = ''

    log_step(
        step='B2', field='booking_status / booking_status_date',
        issue='Ngày YYYY-MM-DD nằm nhầm ở booking_status, booking_status_date rỗng',
        detection_rule='booking_status fullmatch \\d{4}-\\d{2}-\\d{2} AND booking_status_date==""',
        action='Chuyển giá trị sang booking_status_date; để booking_status trống',
        affected_rows=n_swap, changed_cells=n_swap * 2, removed_rows=0,
        example_record_ids=examples_B2,
        reason='Ngày nhầm cột do lỗi nhập liệu; booking_status_date rỗng là bằng chứng đủ',
        check_after='booking_status không còn chứa ngày'
    )

print(f'Đã xử lý {n_swap} dòng.')

In [ ]:
check('B2 booking_status không còn chứa ngày', lambda: not df['booking_status'].astype(str).str.strip().str.fullmatch(r'\d{4}-\d{2}-\d{2}').any())

> **✍️ Quyết định và lý do của nhóm — B2** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Chỉ chuyển khi cả hai điều kiện đều thỏa: `booking_status` có dạng `YYYY-MM-DD` **và** `booking_status_date` đang rỗng, tránh ghi đè dữ liệu đang tồn tại.
> - Vì sao thực hiện B2 trước B4: Nếu chuẩn hóa nhãn trước, ngày nhầm cột sẽ bị xóa thành NaN, mất bằng chứng để phục hồi `booking_status_date`.
> - Ảnh hưởng: Phục hồi `booking_status_date` cho các dòng bị thiếu; `booking_status` của dòng này được để NaN để không đưa ra kết luận sai.


## B3 · Giá trị thiếu theo từng cột
*(Thành viên 1 phụ trách)*

Đưa các ký hiệu thiếu về `pd.NA`; **không điền** bất kỳ giá trị nào.


In [ ]:
# B3 · Chuẩn hóa giá trị thiếu

df = df.astype(object)

MISSING_TOKENS_LOWER = [t.lower() for t in MISSING_TOKENS]
SKIP_COLS = {'record_id', 'reservation_id'}

total_changed = 0
changed_by_col = {}

for col in df.columns:
    if col in SKIP_COLS:
        continue

    before_na = df[col].isna().sum()

    # Ô rỗng → pd.NA
    empty_mask = df[col].astype(str).str.strip() == ''
    df.loc[empty_mask, col] = pd.NA

    # Ký hiệu thiếu → pd.NA
    token_mask = df[col].dropna().astype(str).str.lower().isin(MISSING_TOKENS_LOWER)
    token_idx = df[col].dropna()[token_mask].index
    df.loc[token_idx, col] = pd.NA

    after_na = df[col].isna().sum()
    changed = after_na - before_na
    if changed > 0:
        changed_by_col[col] = changed
        total_changed += changed

# Xử lý Undefined ở market_segment, sales_channel
for col in ['market_segment', 'sales_channel']:
    undef_mask = df[col].dropna().astype(str).str.strip().str.lower() == 'undefined'
    undef_idx = df[col].dropna()[undef_mask].index
    n_undef = len(undef_idx)
    if n_undef > 0:
        df.loc[undef_idx, col] = pd.NA
        changed_by_col[col] = changed_by_col.get(col, 0) + n_undef
        total_changed += n_undef
        print(f'{col}: Undefined → NaN: {n_undef} dòng')

print(f'\nTổng ô thiếu mới: {total_changed}')
print('Phân bổ theo cột:')
for col, cnt in changed_by_col.items():
    print(f'  {col}: {cnt}')

log_step(
    step='B3', field='ALL (trừ record_id, reservation_id)',
    issue='Ký hiệu thiếu đa dạng (NULL, N/A, NA, ?, unknown, -) và ô rỗng',
    detection_rule='strip()=="" OR lower().isin(MISSING_TOKENS)',
    action='Đưa về pd.NA; market_segment/sales_channel Undefined → pd.NA',
    affected_rows=total_changed, changed_cells=total_changed, removed_rows=0,
    reason='Giữ thiếu nguyên bản; không điền mean/mode để tránh sai lệch phân tích',
    check_after=f'Tổng {total_changed} ô thiếu mới'
)

In [ ]:
check('B3 không còn ký hiệu thiếu dạng chữ',
      lambda: not any(df[c].dropna().astype(str).str.lower().isin([t.lower() for t in MISSING_TOKENS]).any()
                      for c in df.columns if c not in ('record_id', 'reservation_id')))

> **✍️ Quyết định và lý do của nhóm — B3** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Đưa tất cả ô rỗng và ký hiệu thiếu (`NULL`, `N/A`, `NA`, `?`, `unknown`, `-`) về `pd.NA` ở mọi cột trừ `record_id` và `reservation_id`. `Undefined` ở `market_segment` và `sales_channel` cũng được đưa về `pd.NA`.
> - Vì sao **không** đưa `Undefined` ở `meal_option` về thiếu: Theo Mục 5.2, `Undefined` và `No Meal` cùng nghĩa — sẽ xử lý bằng MEAL_MAP ở B4.
> - Ảnh hưởng: Các cột `child_passengers`, `passenger_country`, `avg_ticket_fare` có thiếu thực sự giữ nguyên NaN — phân tích sau dùng đúng mẫu số thực tế.


## B4 · Chuẩn hóa nhãn phân loại và nhất quán trạng thái
*(Thành viên 1 phụ trách)*


In [ ]:
# B4 · Chuẩn hóa nhãn phân loại

# --- Bước 1: Strip + gom nhiều khoảng trắng ---
for col in df.select_dtypes(include='object').columns:
    if col == 'record_id':
        continue
    df[col] = df[col].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()) if pd.notna(x) else x)

# --- Hàm ánh xạ ---
def apply_map(series, label_map, col_name):
    key_series = series.apply(clean_key)
    mapped = key_series.map(label_map)
    result = mapped.where(series.notna(), other=pd.NA)
    n_new_na = result.isna().sum() - series.isna().sum()
    if n_new_na > 0:
        print(f'  ⚠️ {col_name}: {n_new_na} giá trị không khớp → NaN')
    return result

# --- Từ điển ánh xạ ---
AIRLINE_MAP = {
    'auroraair': 'Aurora Air', 'aurora': 'Aurora Air',
    'horizonwings': 'Horizon Wings', 'horizon': 'Horizon Wings',
}
MEAL_MAP = {
    'standard': 'Standard', 'vegetarian': 'Vegetarian',
    'special': 'Special', 'nomeal': 'No Meal',
    'undefined': 'No Meal',  # Mục 5.2: Undefined = No Meal
    'none': 'No Meal',
}
SEGMENT_MAP = {
    'direct': 'Direct', 'onlineota': 'Online OTA',
    'offlineagent': 'Offline Agent', 'corporate': 'Corporate',
    'group': 'Group', 'packagetour': 'Package Tour', 'loyalty': 'Loyalty',
}
CHANNEL_MAP = {
    'website': 'Website', 'mobileapp': 'Mobile App',
    'travelagency': 'Travel Agency', 'gdscorporate': 'GDS/Corporate',
    'gds': 'GDS/Corporate', 'corporate': 'GDS/Corporate',
}
FARE_POLICY_MAP = {
    'flexible': 'Flexible', 'semiflexible': 'Semi-Flexible',
    'nonrefundable': 'Non-Refundable',
}
TRAVELER_MAP = {
    'individual': 'Individual', 'linkedbooking': 'Linked Booking',
    'corporatecontract': 'Corporate Contract', 'group': 'Group',
}
STATUS_MAP = {
    'flown': 'Flown', 'completed': 'Flown',
    'cancelled': 'Cancelled', 'canceled': 'Cancelled',
    'noshow': 'No-Show',
}

# --- Áp dụng ánh xạ ---
print('Ánh xạ nhãn phân loại...')
df['airline']        = apply_map(df['airline'], AIRLINE_MAP, 'airline')
df['meal_option']    = apply_map(df['meal_option'], MEAL_MAP, 'meal_option')
df['market_segment'] = apply_map(df['market_segment'], SEGMENT_MAP, 'market_segment')
df['sales_channel']  = apply_map(df['sales_channel'], CHANNEL_MAP, 'sales_channel')
df['fare_policy']    = apply_map(df['fare_policy'], FARE_POLICY_MAP, 'fare_policy')
df['traveler_type']  = apply_map(df['traveler_type'], TRAVELER_MAP, 'traveler_type')
df['booking_status'] = apply_map(df['booking_status'], STATUS_MAP, 'booking_status')

log_step(
    step='B4', field='airline, meal_option, market_segment, sales_channel, fare_policy, traveler_type, booking_status',
    issue='Nhãn phân loại không chuẩn (viết hoa/thường, viết tắt, đồng nghĩa)',
    detection_rule='clean_key(value) không thuộc từ điển chuẩn',
    action='Ánh xạ qua từ điển; giá trị không khớp → pd.NA',
    reason='Chuẩn hóa nhãn để phân tích nhất quán; Completed=Flown theo quy ước nghiệp vụ',
    check_after='Mỗi cột chỉ còn nhãn trong tập chuẩn hoặc NaN'
)

# --- Chuẩn hóa cabin code ---
for cabin_col in ['booked_cabin_code', 'assigned_cabin_code']:
    before_invalid = df[cabin_col].dropna().apply(lambda x: str(x).strip().upper() not in CABIN_CODES).sum()
    df[cabin_col] = df[cabin_col].apply(
        lambda x: str(x).strip().upper() if pd.notna(x) and str(x).strip().upper() in CABIN_CODES else pd.NA
    )
    print(f'{cabin_col}: {before_invalid} mã không hợp lệ → NaN')

log_step(
    step='B4', field='booked_cabin_code, assigned_cabin_code',
    issue='Cabin code chữ thường hoặc ngoài tập hợp lệ',
    detection_rule='upper() not in CABIN_CODES',
    action='Chuyển upper(); mã không hợp lệ → pd.NA',
    reason='Cabin code chuẩn là 2 ký tự: E1-E4, P1-P2, B1-B3, F1-F3',
    check_after='cabin code chỉ có giá trị trong CABIN_CODES hoặc NaN'
)

# --- Chuẩn hóa is_cancelled, is_returning_customer → Int64 0/1 ---
def to_binary_int(series):
    TRUE_VALS  = {'1', '1.0', 'yes', 'true', 'Yes', 'True', 'YES', 'TRUE'}
    FALSE_VALS = {'0', '0.0', 'no', 'false', 'No', 'False', 'NO', 'FALSE'}
    def conv(x):
        if pd.isna(x): return pd.NA
        s = str(x).strip()
        if s in TRUE_VALS:  return 1
        if s in FALSE_VALS: return 0
        try:
            v = float(s)
            if v == 1.0: return 1
            if v == 0.0: return 0
        except: pass
        return pd.NA
    return series.apply(conv).astype('Int64')

df['is_cancelled'] = to_binary_int(df['is_cancelled'])
df['is_returning_customer'] = to_binary_int(df['is_returning_customer'])
print(f'\nis_cancelled dtype: {df["is_cancelled"].dtype}')
print(f'is_returning_customer dtype: {df["is_returning_customer"].dtype}')

log_step(
    step='B4', field='is_cancelled, is_returning_customer',
    issue='Giá trị yes/no/True/False/0.0/1.0 thay vì số nguyên 0/1',
    detection_rule='value không phải 0 hoặc 1 nguyên',
    action='Chuyển về Int64 0/1; giá trị không rõ → pd.NA',
    reason='Hai cột cờ phải là số nguyên; Int64 cho phép NaN',
    check_after='is_cancelled, is_returning_customer dtype == Int64'
)

# --- Tính lại is_cancelled từ booking_status hợp lệ ---
status_flown_mask = df['booking_status'] == 'Flown'
status_cancel_mask = df['booking_status'].isin(['Cancelled', 'No-Show'])

contradict_before = (
    (status_flown_mask & (df['is_cancelled'] == 1)) |
    (status_cancel_mask & (df['is_cancelled'] == 0))
).sum()

df.loc[status_flown_mask, 'is_cancelled'] = pd.array([0] * status_flown_mask.sum(), dtype='Int64')
df.loc[status_cancel_mask, 'is_cancelled'] = pd.array([1] * status_cancel_mask.sum(), dtype='Int64')

print(f'\nMâu thuẫn is_cancelled vs booking_status: {contradict_before} dòng đã được sửa')

log_step(
    step='B4', field='is_cancelled',
    issue=f'is_cancelled mâu thuẫn với booking_status ({contradict_before} dòng)',
    detection_rule='(Flown AND is_cancelled=1) OR (Cancelled/No-Show AND is_cancelled=0)',
    action='Khi booking_status hợp lệ: Flown→0, Cancelled/No-Show→1',
    affected_rows=contradict_before, changed_cells=contradict_before,
    reason='booking_status là bằng chứng nghiệp vụ đáng tin cậy hơn cờ is_cancelled',
    check_after='is_cancelled nhất quán với booking_status'
)

In [ ]:
check('B4 airline chỉ còn Aurora Air / Horizon Wings và không thiếu', lambda: set(df['airline'].dropna()) <= set(AIRLINES) and df['airline'].notna().all())
check('B4 meal_option hợp lệ hoặc thiếu', lambda: df['meal_option'].dropna().isin(MEALS).all())
check('B4 market_segment hợp lệ hoặc thiếu', lambda: df['market_segment'].dropna().isin(SEGMENTS).all())
check('B4 sales_channel hợp lệ hoặc thiếu', lambda: df['sales_channel'].dropna().isin(CHANNELS).all())
check('B4 fare_policy hợp lệ hoặc thiếu', lambda: df['fare_policy'].dropna().isin(FARE_POLICIES).all())
check('B4 traveler_type hợp lệ hoặc thiếu', lambda: df['traveler_type'].dropna().isin(TRAVELERS).all())
check('B4 booking_status hợp lệ hoặc thiếu', lambda: df['booking_status'].dropna().isin(STATUSES).all())
check('B4 cabin code hợp lệ hoặc thiếu', lambda: df['booked_cabin_code'].dropna().isin(CABIN_CODES).all() and df['assigned_cabin_code'].dropna().isin(CABIN_CODES).all())
check('B4 hai cờ 0/1 kiểu Int64', lambda: str(df['is_cancelled'].dtype) == 'Int64' and str(df['is_returning_customer'].dtype) == 'Int64')

> **✍️ Quyết định và lý do của nhóm — B4** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Dùng hàm `clean_key()` (loại ký tự không phải chữ cái, chuyển thường) làm khóa để ánh xạ nhãn, bắt được mọi biến thể viết hoa/thường/dấu cách. `Undefined` ở `meal_option` → `No Meal` theo Mục 5.2. `Completed` → `Flown` theo quy ước nghiệp vụ.
> - Vì sao không để nhãn không khớp thành NaN cho `airline`: Tất cả biến thể đều là viết sai/viết tắt của Aurora Air hoặc Horizon Wings — có thể ánh xạ đầy đủ.
> - Ảnh hưởng: Sau B4, `booking_status` và `is_cancelled` hoàn toàn nhất quán, đảm bảo tỷ lệ hủy tính đúng ở các câu phân tích.


## B5 · Ngày khởi hành và ngày cập nhật trạng thái
*(Thành viên 1 phụ trách)*

Chuyển `departure_date` (4 định dạng) và `booking_status_date` sang kiểu datetime chuẩn.


In [ ]:
# B5 · Chuyển đổi định dạng ngày tháng

# Bốn định dạng của departure_date theo đề bài
DATE_FORMATS = [
    '%Y-%m-%d',    # 2024-11-06 (ISO)
    '%d/%m/%Y',    # 23/08/2025 (ngày/tháng/năm - ghi rõ dayfirst)
    '%B %d, %Y',   # October 6, 2025
    '%d-%b-%Y',    # 30-Jun-2025
    '%d %b %Y',    # 15 Nov 2024
]

# --- Parse departure_date ---
dep_str = df['departure_date'].copy()
dep_parsed = pd.Series([pd.NaT] * len(df), index=df.index, dtype='datetime64[ns]')

for fmt in DATE_FORMATS:
    remaining_mask = dep_parsed.isna() & dep_str.notna()
    if remaining_mask.sum() == 0:
        break
    parsed_attempt = pd.to_datetime(
        dep_str.where(remaining_mask, other=None),
        format=fmt, errors='coerce'
    )
    dep_parsed = dep_parsed.fillna(parsed_attempt)
    n_newly_parsed = (~dep_parsed.isna() & remaining_mask).sum()
    if n_newly_parsed > 0:
        print(f'Format {fmt}: parse thêm {n_newly_parsed} dòng')

# Loại bỏ ngày ngoài miền
out_of_range_dep = dep_parsed.notna() & ~dep_parsed.between(DEPARTURE_MIN, DEPARTURE_MAX)
n_out_dep = out_of_range_dep.sum()
dep_parsed[out_of_range_dep] = pd.NaT
df['departure_date'] = dep_parsed

print(f'\ndeparture_date: {df["departure_date"].notna().sum()} hợp lệ, '
      f'{df["departure_date"].isna().sum()} thiếu/sai, {n_out_dep} ngoài miền → NaT')

log_step(
    step='B5', field='departure_date',
    issue='Nhiều định dạng ngày; ngày ngoài miền 2024-07-01 đến 2026-08-31',
    detection_rule='pd.to_datetime(format=..., errors="coerce"); not between(DEPARTURE_MIN, DEPARTURE_MAX)',
    action='Parse từng format tuần tự; ngày ngoài miền → NaT',
    affected_rows=int(df['departure_date'].isna().sum()),
    reason='Chuẩn hóa về datetime; không suy đoán ngày sai',
    check_after=f'{df["departure_date"].notna().sum()} hợp lệ'
)

# --- Parse booking_status_date ---
bsd_str = df['booking_status_date'].copy()
bsd_parsed = pd.to_datetime(bsd_str, format='%Y-%m-%d', errors='coerce')

out_of_range_bsd = bsd_parsed.notna() & ~bsd_parsed.between(STATUS_MIN, STATUS_MAX)
n_out_bsd = out_of_range_bsd.sum()
bsd_parsed[out_of_range_bsd] = pd.NaT
df['booking_status_date'] = bsd_parsed

print(f'booking_status_date: {df["booking_status_date"].notna().sum()} hợp lệ, '
      f'{df["booking_status_date"].isna().sum()} thiếu/sai, {n_out_bsd} ngoài miền → NaT')

log_step(
    step='B5', field='booking_status_date',
    issue='Ngày ngoài miền 2024-01-01 đến 2026-09-15 hoặc không đọc được',
    detection_rule='pd.to_datetime(format="%Y-%m-%d"); not between(STATUS_MIN, STATUS_MAX)',
    action='Parse ISO; ngày ngoài miền → NaT',
    affected_rows=int(df['booking_status_date'].isna().sum()),
    reason='Chỉ ISO YYYY-MM-DD là định dạng chuẩn của cột này',
    check_after=f'{df["booking_status_date"].notna().sum()} hợp lệ'
)

In [ ]:
check('B5 departure_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['departure_date'])
      and df['departure_date'].dropna().between(DEPARTURE_MIN, DEPARTURE_MAX).all())
check('B5 booking_status_date là kiểu datetime, trong miền', lambda: pd.api.types.is_datetime64_any_dtype(df['booking_status_date'])
      and df['booking_status_date'].dropna().between(STATUS_MIN, STATUS_MAX).all())

> **✍️ Quyết định và lý do của nhóm — B5** *(Thành viên 1)*
>
> - Quy tắc áp dụng: Parse `departure_date` tuần tự theo 5 định dạng (ISO → DD/MM/YYYY → Month DD, YYYY → DD-Mon-YYYY), dùng `fillna` để ghép kết quả. Không để pandas tự đoán để tránh nhầm `05/03/2025` (tháng 3 hay tháng 5).
> - Vì sao không sửa ngày sai thành ngày gần nhất: Không có bằng chứng để xác định ngày đúng; giữ NaT và ghi nhật ký là trung thực nhất.
> - Ảnh hưởng: Sau B5, các cột ngày dùng được cho tính `total_trip_days`, `departure_month`, `departure_year` ở C1.


---
## B6 · Số đo: lead time, số ngày chuyến đi, số hành khách và giá vé
*(Thành viên 2 phụ trách)*


In [ ]:
# TODO B6 — Thành viên 2 phụ trách
# Xử lý: booking_lead_days (0-365), trip_weekend_days (0-20), trip_weekdays (0-50)
#         adult/child/infant_passengers (0-10), special_service_requests (0-5)
#         avg_ticket_fare: bỏ $, USD, xử lý dấu phẩy thập phân, miền 0-5000
pass

## B7 · Chuẩn hóa mã quốc gia
*(Thành viên 2 phụ trách)*


In [ ]:
# TODO B7 — Thành viên 2 phụ trách
# Chuẩn hóa passenger_country: in hoa 2-3 chữ cái, map CN→CHN v.v.
pass

## B8 · Hợp nhất bản ghi và lập bảng truy vết row_lineage
*(Thành viên 2 phụ trách)*

**Lưu ý quan trọng:** Sử dụng biến `dropped_record_ids_B1` từ B1 khi lập `row_lineage`.


In [ ]:
# TODO B8 — Thành viên 2 phụ trách
# Hợp nhất các dòng cùng reservation_id (~10.000 dòng sạch)
# Lập row_lineage.csv đủ 10.325 dòng gốc (kept/merged/dropped)
# dropped_record_ids_B1 chứa record_id đã bị loại ở B1
pass

---
# GIAI ĐOẠN C · BẢNG SẠCH & KIỂM TRA
*(Thành viên 2 phụ trách C1–C4)*


In [ ]:
# TODO C1 — Thành viên 2
# Tạo cột dẫn xuất:
# - departure_year, departure_month (từ departure_date)
# - total_trip_days = trip_weekend_days + trip_weekdays
# - total_passengers = adult + child + infant
# - lead_time_group: '0-7', '8-30', '31-90', '91-180', '181+'
# - cabin_changed_flag: 1 nếu booked_cabin_code != assigned_cabin_code, 0 nếu bằng nhau
# - booking_value_est = avg_ticket_fare * total_passengers
pass

In [ ]:
# TODO C2 — Thành viên 2
# Tạo monthly_summary.csv
# Tỷ lệ hủy = tổng số hủy / tổng số có trạng thái xác định * 100
# KHÔNG lấy trung bình cộng tỷ lệ 12 tháng
pass

In [ ]:
# TODO C3 — Thành viên 2
# Xuất 5 tệp dữ liệu cốt lõi:
# reservations_clean.csv, monthly_summary.csv, cleaning_log.csv,
# row_lineage.csv, quality_report.csv
pass

In [ ]:
# TODO C4 — Thành viên 2
# Chạy ô kiểm tra tổng thể
pass

---
# GIAI ĐOẠN D · 10 CÂU HỎI PHÂN TÍCH

## Câu 1 · So sánh chất lượng trước/sau làm sạch
*(Thành viên 2 phụ trách)*


In [ ]:
# TODO Câu 1 — Thành viên 2
# Xuất q01_before_after.csv
pass

## Câu 2–10 · Phân tích nghiệp vụ
*(Thành viên 3 phụ trách)*

**Lưu ý quan trọng:**
- Câu 8 (giá vé) và Câu 9 (số ngày): **Chỉ tính đặt chỗ KHÔNG HỦY** (`is_cancelled == 0`)
- Câu 10 (đổi cabin): **Chỉ tính đặt chỗ đã bay** (`booking_status == 'Flown'`)
- Tỷ lệ hủy = số hủy / tổng có trạng thái xác định × 100 (không lấy mean của các tỷ lệ)


In [ ]:
# TODO Câu 2 — Thành viên 3: Tỷ lệ hủy theo hãng → q02_airline_cancel.csv + fig01_airline_cancel.png
pass

In [ ]:
# TODO Câu 3 — Thành viên 3: Mùa vụ đặt chỗ năm 2025 → q03_monthly_departures_2025.csv + fig02_monthly_departures_2025.png
pass

In [ ]:
# TODO Câu 4 — Thành viên 3: Lead time và tỷ lệ hủy → q04_leadtime_stats.csv, q04_leadtime_cancel.csv + fig03_leadtime_cancel.png
pass

In [ ]:
# TODO Câu 5 — Thành viên 3: Top 10 quốc gia đặt chỗ → q05_top10_country.csv + fig04_top10_country.png
pass

In [ ]:
# TODO Câu 6 — Thành viên 3: Phân khúc thị trường → q06_segment_cancel.csv + fig05_segment_cancel.png
pass

In [ ]:
# TODO Câu 7 — Thành viên 3: Chính sách giá và trạng thái → q07_fare_policy_status.csv + fig06_fare_policy_status.png
pass

In [ ]:
# TODO Câu 8 — Thành viên 3: Phân phối giá vé (chỉ không hủy) → q08_fare_stats_by_airline.csv + fig07_fare_hist.png
pass

In [ ]:
# TODO Câu 9 — Thành viên 3: Phân phối số ngày chuyến đi (chỉ không hủy) → q09_trip_days.csv + fig08_trip_days.png
pass

In [ ]:
# TODO Câu 10 — Thành viên 3: Tỷ lệ đổi cabin (chỉ Flown) → q10_cabin_change_by_airline.csv + q10_cabin_change_by_booked_code.csv
pass

---
# GIAI ĐOẠN E · TIẾN ĐỘ VÀ NỘP BÀI


In [ ]:
# Kiểm tra tiến độ tổng thể
show_progress()